ĐỀ BÀI: 

- Thống kê các luồng thông tin, đồn đoán liên quan vị trí Thủ tướng kế nhiệm sắp tới trên các nền tảng MXH (Nêu rõ đồn đoán bao nhiêu ông, các luồng thông tin, nội dung liên quan mỗi ông theo tích cực/tiêu cực; và gom nhóm thành các nhóm vấn đề; nhóm đối tượng/trang kênh/tài khoản nào tán phát với tần suất cao, đáng chú ý; phân bổ theo nền tảng...).

- Tổng hợp các phản ứng, bình luận của dư luận liên quan các luồng thông tin trên.
```
1. Trình bày bằng file excel / thống kê qua các bảng biểu
2. Ứng viên tự tổng hợp dữ liệu để đánh giá, phân tích).


Chọn bối cảnh là năm 2026 và trong nước Việt Nam. Vì mục tiêu yêu cầu thống kê dữ liệu thông tin lời đồn đoán nên tập trung cào dữ liệu từ 2 nền tảng có nhiều phân khúc người dùng nhất hiện nay là Youtube và Facebook

I. QUÉT VÀ LÀM SẠCH DỮ LIỆU

In [ ]:
# 1. IMPORT THƯ VIỆN
import pandas as pd
import re
from datetime import datetime
from apify_client import ApifyClient
import time
import json

Sau khi tạo thư viện cần thiết cho xử lý dữ liệu và cào dữ liệu từ google về, bây giờ tiến hành lựa chọn từ khóa và setup cào dữ liệu:

Bộ từ khóa được setup như sau:
- Bộ từ khóa "thủ tướng kế nhiệm Việt Nam", "Kiện toàn", "Nhân sự cấp cao",..: Trong báo chí chính thống Việt Nam, đây là các thuật ngữ chuyên môn (Jargon) thay thế cho từ "thay đổi lãnh đạo". Dùng từ này giúp lọc được các bài phân tích sâu thay vì tin tức thời sự vụn vặt.

In [ ]:
# 2. THỰC HIỆN QUÉT APIFY VỚI BỘ TỪ KHÓA
# Cấu hình API Apify
API_KEY = "YOUR_APIFY_API_KEY_HERE"
client = ApifyClient(API_TOKEN)

# Bộ từ khóa quét Version 2 (Mở rộng đa chiều: Tích cực - Trung lập - Tiêu cực)
queries = [
    # Nhóm 1: Quét trực diện vào các đồn đoán nhân sự (Thường có nhiều tin hành lang)
    'site:youtube.com "ai làm thủ tướng" 2026',
    'site:facebook.com "ghế thủ tướng" khóa 14',
    'site:youtube.com "phù thủy" nhân sự Việt Nam 2026',
    'site:facebook.com "sắp xếp" tứ trụ 2026',
    
    # Nhóm 2: Quét theo tên ứng viên cụ thể kèm từ lóng (Để bắt thành phần tiêu cực)
    'site:youtube.com "Lê Minh Hưng" "Trương Mỹ Lan" "SCB"',
    'site:facebook.com "Lê Minh Hưng" "kế nhiệm" "thủ tướng"',
    'site:youtube.com "Tô Lâm" "quyền lực" "thâu tóm"',
    'site:facebook.com "Phạm Minh Chính" "nghỉ" "tiếp tục"',
    
    # Nhóm 3: Quét các từ khóa về phe phái/đấu đá (Yêu cầu đặc biệt của Agency)
    'site:youtube.com "đấu đá" nhân sự cấp cao',
    'site:facebook.com "phe phái" Đại hội XIV',
    'site:youtube.com "thanh trừng" nội bộ 2026',
    '"tin đồn" nhân sự bộ chính trị site:facebook.com',
    
    # Nhóm 4: Quét rộng về tương lai kinh tế gắn liền với nhân sự
    'site:youtube.com "vận mệnh" Việt Nam 2026',
    'site:facebook.com "đổi mới" nhân sự chính phủ'
]
print(" ĐANG LẤY LINK TỪ GOOGLE!")

run_input = {
    "queries": "\n".join(queries),
    "maxPagesPerQuery": 2, 
    "resultsPerPage": 20, 
    "mobileResults": False,
    "languageCode": "vi",
    "locationCode": "VN"
}


# Chạy Actor Search
run = client.actor("apify/google-search-scraper").call(
    run_input=run_input,
    memory_mbytes=512
)
search_results = [item for item in client.dataset(run["defaultDatasetId"]).iterate_items()]
all_links = []
for page in search_results:
    for result in page.get('organicResults', []):
        url = result.get('url')
        if url:
            all_links.append({
                "Title": result.get('title'),
                "URL": url,
                "Description": result.get('description', '')
            })
# Chuyển thành DataFrame và xóa trùng URL
df_results = pd.DataFrame(all_links).drop_duplicates(subset=['URL'])
print(f"QUÉT ĐƯỢC {len(df_results)} LINK!")
# Lưu file trung gian
df_results.to_csv("MAX_LINKS_2026.csv", index=False, encoding='utf-8-sig')

# Danh sách từ khóa lọc lại kết quả (Bổ sung từ lóng và từ khóa tiêu cực)
keywords = [
    # Từ khóa chính thống
    'nhân sự', 'kế nhiệm', 'kiện toàn', 'Đại hội XIV', '2026', 'thủ tướng', 'chủ tịch', 
    # Từ khóa về hành vi/thái độ (Để bắt được nội dung có giá trị)
    'đồn đoán', 'tin hành lang', 'bí mật', 'hé lộ', 'phân tích', 'dự báo',
    # Từ khóa tiêu cực/nhạy cảm (Bắt thành phần tiêu cực theo yêu cầu)
    'đấu đá', 'phe phái', 'thanh trừng', 'hạ bệ', 'sai phạm', 'trách nhiệm', 
    'sân sau', 'lợi ích nhóm', 'ngã ngựa', 'về vườn', 'từ chức', 'bắt giam'
]

# Thực hiện lọc lại với bộ keyword mới
df_filtered = df_results[
    df_results['Title'].str.contains('|'.join(keywords), case=False, na=False) | 
    df_results['Description'].str.contains('|'.join(keywords), case=False, na=False)
].copy()
print(f"SAU KHI LỌC KEYWORD, CÒN LẠI {len(df_filtered)} LINK.")

# Tách riêng link YouTube
df_youtube = df_filtered[df_filtered['URL'].str.contains('youtube.com|youtu.be', case=False, na=False)]
df_youtube.to_csv("YOUTUBE_LINKS_2026.csv", index=False, encoding='utf-8-sig')

# Tách link Facebook
df_facebook = df_filtered[~df_filtered['URL'].str.contains('youtube.com|youtu.be', case=False, na=False)]
df_facebook.to_csv("FACEBOOK_LINKS_2026.csv", index=False, encoding='utf-8-sig')


Sau khi cào dữ liệu liệu bằng cách thông qua APIFY để lấy dữ liệu từ Google, tiến hành cào dữ liệu riêng cho từng loại nền tảng, ở đây bao gồm: facebook và youtube bằng tool APIFY.

In [ ]:
# 3. TỪ ĐIỂN MỞ RỘNG (Đồng bộ với YouTube)
SENTIMENT_DICT = {
    'Positive': [
        'ủng hộ', 'tin tưởng', 'xứng đáng', 'tuyệt vời', 'quyết liệt', 'bản lĩnh', 
        'tài đức', 'liêm chính', 'trong sạch', 'vì dân', 'gần dân', 'tận tâm', 
        'người đốt lò', 'vững vàng', 'kỳ vọng', 'đổi mới', 'bứt phá', 'tâm huyết', 'ngưỡng mộ'
    ],
    'Negative': [
    # Nhóm 1: Sai phạm & Hình sự (Củi lửa)
    'tham nhũng', 'hối lộ', 'lợi ích nhóm', 'sân sau', 'sai phạm', 'vơ vét', 'thất thoát', 
    'lãng phí', 'biến thủ', 'tư lợi', 'nhũng nhiễu', 'cửa quyền', 'hách dịch', 'tha hóa', 
    'khởi tố', 'bắt giam', 'xộ khám', 'nhập kho', 'dính chàm', 'nhúng chàm', 'kỷ luật',
    
    # Nhóm 2: Đấu đá & Thuyết âm mưu (Đặc trưng đồn đoán nhân sự)
    'đấu đá', 'phe phái', 'thanh trừng', 'hạ bệ', 'thâu tóm', 'quyền lực ảo', 'soán ngôi', 
    'đảo chính', 'mất đoàn kết', 'tranh giành', 'ghế nóng', 'triệt hạ', 'vận động hành lang',
    'quân xanh quân đỏ', 'cài cắm', 'thân hữu',
    
    # Nhóm 3: Từ lóng & Mỉa mai (Dân mạng hay dùng)
    'ngã ngựa', 'về vườn', 'hạ cánh an toàn', 'bay màu', 'quay xe', 'diễn kịch', 'màu mè', 
    'mị dân', 'bù nhìn', 'bốc phét', 'tào lao', 'vớ vẩn', 'lùa gà', 'múa rìu', 'làm màu',
    'nịnh bợ', 'thổi phồng', 'ảo tưởng', 'hết thời', 'xuống hố',
    
    # Nhóm 4: Sự bất mãn & Chỉ trích năng lực
    'yếu kém', 'bất tài', 'thất vọng', 'phản đối', 'không xứng', 'loạn', 'suy thoái', 
    'xuống cấp', 'tệ hại', 'trắc trở', 'khó khăn', 'bế tắc', 'vô trách nhiệm', 'bao che'
    ]
}

FIELD_DICT = {
    'Nhân sự - Quyền lực': ['bộ chính trị', 'ban bí thư', 'khóa 14', 'kế nhiệm', 'quyền lực', 'tứ trụ', 'bầu cử', 'nhân sự đại hội', 'sắp xếp ghế'],
    'Kinh tế - Tài chính': ['kinh tế', 'ngân hàng', 'scb', 'trương mỹ lan', 'vạn thịnh phát', 'lạm phát', 'thống đốc', 'tài chính', 'tiền tệ'],
    'An ninh - Nội chính': ['công an', 'điều tra', 'bắt giam', 'đốt lò', 'an ninh', 'vi phạm pháp luật', 'tội phạm', 'khởi tố'],
    'Uy tín - Đời tư': ['đạo đức', 'lối sống', 'trong sạch', 'biệt phủ', 'con ông cháu cha', 'tâm huyết', 'gần dân', 'giản dị', 'có tâm', 'có tầm']
}

# CÁC HÀM XỬ LÝ
def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]', ' ', text)
    return " ".join(text.split()).strip()

def get_precise_sentiment_and_keywords(text):
    text = f" {text} "
    matched_pos = [w for w in SENTIMENT_DICT['Positive'] if w in text]
    matched_neg = [w for w in SENTIMENT_DICT['Negative'] if w in text]
    
    res = []
    if matched_pos: res.append(f"[+] {', '.join(matched_pos)}")
    if matched_neg: res.append(f"[-] {', '.join(matched_neg)}")
    all_keywords = " | ".join(res) if res else "Không có từ khóa tiêu biểu"
    
    if len(matched_neg) > len(matched_pos): return 'Tiêu cực', all_keywords
    if len(matched_pos) > len(matched_neg): return 'Tích cực', all_keywords
    return 'Trung lập', all_keywords

def get_precise_field(text):
    text = f" {text} "
    scores = {field: 0 for field in FIELD_DICT.keys()}
    for field, keywords in FIELD_DICT.items():
        for word in keywords:
            if word in text:
                scores[field] += 1
    max_field = max(scores, key=scores.get)
    return max_field if scores[max_field] > 0 else 'Khác'

def get_precise_candidate_info(text):
    text = f" {text} "
    mapping = [
        ('Lê Minh Hưng', r'lê minh hưng|ông hưng|thống đốc hưng', 'Thủ tướng'),
        ('Phạm Minh Chính', r'phạm minh chính|thủ tướng chính|ông chính', 'Thủ tướng (Tái cử)'),
        ('Tô Lâm', r'tô lâm|tổng bí thư|bác lâm', 'Tổng Bí thư / Chủ tịch nước'),
        ('Lương Cường', r'lương cường|ông cường|đại tướng cường', 'Chủ tịch nước'),
        ('Trần Thanh Mẫn', r'trần thanh mẫn|ông mẫn|chủ tịch mẫn', 'Chủ tịch Quốc hội'),
        ('Nguyễn Hòa Bình', r'nguyễn hòa bình|ông bình|phó thủ tướng bình', 'Phó Thủ tướng')
    ]
    for name, pattern, position in mapping:
        if re.search(pattern, text):
            return name, position
    return 'Chưa xác định', 'N/A'

# THỰC THI CHO FACEBOOK
file_path = r"D:\DATA1\DA_NTH\Interview_test\Keyagency\dataset_facebook-comments-scraper_2026-03-24_08-35-38-680.csv" 
df_raw = pd.read_csv(file_path)

df_fb = pd.DataFrame()
# Facebook Scraper thường để nội dung ở cột 'text'
df_fb['Content_Raw'] = df_raw.get('text', "")
df_fb['Content_Cleaned'] = df_fb['Content_Raw'].apply(clean_text)

# Áp dụng logic trích xuất
df_fb['Candidate'], df_fb['Target_Position'] = zip(*df_fb['Content_Cleaned'].apply(get_precise_candidate_info))
df_fb['Field'] = df_fb['Content_Cleaned'].apply(get_precise_field)
df_fb['Sentiment'], df_fb['Keyword'] = zip(*df_fb['Content_Cleaned'].apply(get_precise_sentiment_and_keywords))

# Cấu hình thông tin nguồn
df_fb['Source_Platform'] = 'Facebook'
df_fb['URL'] = df_raw.get('facebookUrl', "N/A")
df_fb['Like_count'] = pd.to_numeric(df_raw.get('likesCount', 0), errors='coerce').fillna(0)

# SẮP XẾP VÀ LỌC
column_order = [
    'Source_Platform', 'URL', 'Content_Raw', 'Content_Cleaned', 
    'Like_count', 'Sentiment', 'Candidate', 'Target_Position', 'Field', 'Keyword'
]

df_fb_final = df_fb[df_fb['Candidate'] != 'Chưa xác định'][column_order]

# Lưu file official
df_fb_final.to_csv("facebook_official_v4.csv", index=False, encoding='utf-8-sig')

print(f"Xử lý xong Facebook! Đã trích xuất {len(df_fb_final)} dòng thảo luận về nhân sự.")


In [ ]:
# 4 ĐỊNH NGHĨA TỪ ĐIỂN MỞ RỘNG
SENTIMENT_DICT = {
    'Positive': [
        'ủng hộ', 'tin tưởng', 'xứng đáng', 'tuyệt vời', 'quyết liệt', 'bản lĩnh', 
        'tài đức', 'liêm chính', 'trong sạch', 'vì dân', 'gần dân', 'tận tâm', 
        'người đốt lò', 'vững vàng', 'kỳ vọng', 'đổi mới', 'bứt phá', 'tâm huyết'
    ],
    'Negative': [
    # Nhóm 1: Sai phạm & Hình sự (Củi lửa)
    'tham nhũng', 'hối lộ', 'lợi ích nhóm', 'sân sau', 'sai phạm', 'vơ vét', 'thất thoát', 
    'lãng phí', 'biến thủ', 'tư lợi', 'nhũng nhiễu', 'cửa quyền', 'hách dịch', 'tha hóa', 
    'khởi tố', 'bắt giam', 'xộ khám', 'nhập kho', 'dính chàm', 'nhúng chàm', 'kỷ luật',
    
    # Nhóm 2: Đấu đá & Thuyết âm mưu (Đặc trưng đồn đoán nhân sự)
    'đấu đá', 'phe phái', 'thanh trừng', 'hạ bệ', 'thâu tóm', 'quyền lực ảo', 'soán ngôi', 
    'đảo chính', 'mất đoàn kết', 'tranh giành', 'ghế nóng', 'triệt hạ', 'vận động hành lang',
    'quân xanh quân đỏ', 'cài cắm', 'thân hữu',
    
    # Nhóm 3: Từ lóng & Mỉa mai (Dân mạng hay dùng)
    'ngã ngựa', 'về vườn', 'hạ cánh an toàn', 'bay màu', 'quay xe', 'diễn kịch', 'màu mè', 
    'mị dân', 'bù nhìn', 'bốc phét', 'tào lao', 'vớ vẩn', 'lùa gà', 'múa rìu', 'làm màu',
    'nịnh bợ', 'thổi phồng', 'ảo tưởng', 'hết thời', 'xuống hố',
    
    # Nhóm 4: Sự bất mãn & Chỉ trích năng lực
    'yếu kém', 'bất tài', 'thất vọng', 'phản đối', 'không xứng', 'loạn', 'suy thoái', 
    'xuống cấp', 'tệ hại', 'trắc trở', 'khó khăn', 'bế tắc', 'vô trách nhiệm', 'bao che'
    ]
}

FIELD_DICT = {
    'Nhân sự - Quyền lực': ['bộ chính trị', 'ban bí thư', 'khóa 14', 'kế nhiệm', 'quyền lực', 'tứ trụ', 'bầu cử', 'nhân sự đại hội', 'sắp xếp ghế'],
    'Kinh tế - Tài chính': ['kinh tế', 'ngân hàng', 'scb', 'trương mỹ lan', 'vạn thịnh phát', 'lạm phát', 'thống đốc', 'tài chính', 'tiền tệ'],
    'An ninh - Nội chính': ['công an', 'điều tra', 'bắt giam', 'đốt lò', 'an ninh', 'vi phạm pháp luật', 'tội phạm', 'khởi tố'],
    'Uy tín - Đời tư': ['đạo đức', 'lối sống', 'trong sạch', 'biệt phủ', 'con ông cháu cha', 'tâm huyết', 'gần dân', 'giản dị', 'có tâm', 'có tầm']
}

# CÁC HÀM XỬ LÝ LOGIC NÂNG CẤP
def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]', ' ', text)
    return " ".join(text.split()).strip()

def get_precise_sentiment_and_keywords(text):
    text = f" {text} "
    matched_pos = [w for w in SENTIMENT_DICT['Positive'] if w in text]
    matched_neg = [w for w in SENTIMENT_DICT['Negative'] if w in text]
    
    # Tạo chuỗi Keyword
    res = []
    if matched_pos: res.append(f"{', '.join(matched_pos)}")
    if matched_neg: res.append(f"{', '.join(matched_neg)}")
    all_keywords = " | ".join(res) if res else "Không có từ khóa tiêu biểu"
    
    # Phân loại Sentiment dựa trên trọng số
    if len(matched_neg) > len(matched_pos): return 'Tiêu cực', all_keywords
    if len(matched_pos) > len(matched_neg): return 'Tích cực', all_keywords
    return 'Trung lập', all_keywords

def get_precise_field(text):
    text = f" {text} "
    scores = {field: 0 for field in FIELD_DICT.keys()}
    for field, keywords in FIELD_DICT.items():
        for word in keywords:
            if word in text:
                scores[field] += 1
    
    max_field = max(scores, key=scores.get)
    return max_field if scores[max_field] > 0 else 'Khác'

def get_precise_candidate_info(text):
    text = f" {text} "
    # Sử dụng r'\b' để đảm bảo bắt chính xác từ, không bắt dính vào từ khác
    mapping = [
        ('Lê Minh Hưng', r'lê minh hưng|ông hưng|thống đốc hưng', 'Thủ tướng'),
        ('Phạm Minh Chính', r'phạm minh chính|thủ tướng chính|ông chính', 'Thủ tướng (Tái cử)'),
        ('Tô Lâm', r'tô lâm|tổng bí thư|bác lâm', 'Tổng Bí thư / Chủ tịch nước'),
        ('Lương Cường', r'lương cường|ông cường|đại tướng cường', 'Chủ tịch nước'),
        ('Trần Thanh Mẫn', r'trần thanh mẫn|ông mẫn|chủ tịch mẫn', 'Chủ tịch Quốc hội'),
        ('Nguyễn Hòa Bình', r'nguyễn hòa bình|ông bình|phó thủ tướng bình', 'Phó Thủ tướng')
    ]
    for name, pattern, position in mapping:
        if re.search(pattern, text):
            return name, position
    return 'Chưa xác định', 'N/A'

# THỰC THI VỚI YOUTUBE
file_name = r"D:\DATA1\DA_NTH\Interview_test\Keyagency\dataset_youtube-comments-scraper_2026-03-24_07-56-34-035.csv"
df_raw = pd.read_csv(file_name)
df_raw.columns = [str(c).strip() for c in df_raw.columns]

df_final = pd.DataFrame()
df_final['Content_Raw'] = df_raw.get('comment', "")
df_final['Content_Cleaned'] = df_final['Content_Raw'].apply(clean_text)

# Áp dụng các hàm trích xuất
df_final['Candidate'], df_final['Target_Position'] = zip(*df_final['Content_Cleaned'].apply(get_precise_candidate_info))
df_final['Field'] = df_final['Content_Cleaned'].apply(get_precise_field)
df_final['Sentiment'], df_final['Keyword'] = zip(*df_final['Content_Cleaned'].apply(get_precise_sentiment_and_keywords))

# Cấu hình các cột phụ trợ
df_final['Source_Platform'] = 'YouTube'
df_final['URL'] = df_raw.get('pageUrl', "N/A")
df_final['Like_count'] = pd.to_numeric(df_raw.get('voteCount', 0), errors='coerce').fillna(0)

# SẮP XẾP VÀ LỌC DỮ LIỆU
column_order = [
    'Source_Platform', 'URL', 'Content_Raw', 'Content_Cleaned', 
    'Like_count', 'Sentiment', 'Candidate', 'Target_Position', 'Field', 'Keyword'
]

# Lọc bỏ nhiễu và lưu file
df_output = df_final[df_final['Candidate'] != 'Chưa xác định'][column_order]
df_output.to_csv("youtube_final_precision_v4.csv", index=False, encoding='utf-8-sig')

print(f"Hoàn thành! Đã lọc được {len(df_output)} dòng dữ liệu nhân sự chất lượng.")

In [ ]:
# 5 ĐỌC 2 FILE DỮ LIỆU ĐÃ XỬ LÝ
df_fb = pd.read_csv(r'D:\DATA1\DA_NTH\Interview_test\Keyagency\Result_Cleaned\facebook_official.csv')
df_yt = pd.read_csv(r'D:\DATA1\DA_NTH\Interview_test\Keyagency\Result_Cleaned\youtube_official.csv')

# GỘP 2 FILE THÀNH MASTER DATASET
df_master = pd.concat([df_fb, df_yt], ignore_index=True)

# HÀM TÁCH KEYWORD TÍCH CỰC VÀ TIÊU CỰC
def split_keywords(keyword_str):
    if pd.isna(keyword_str) or keyword_str == "Không có từ khóa tiêu biểu" or keyword_str == "":
        return "Không có", "Không có"
    
    pos_keywords = "Không có"
    neg_keywords = "Không có"
    
    # Tách chuỗi dựa trên dấu phân cách " | " nếu có cả 2 loại
    parts = str(keyword_str).split(" | ")
    
    for part in parts:
        if "[+]" in part:
            pos_keywords = part.replace("[+]", "").strip()
        elif "[-]" in part:
            neg_keywords = part.replace("[-]", "").strip()
            
    return pos_keywords, neg_keywords

# ÁP DỤNG TÁCH CỘT
df_master['Keyword_Tich_Cuc'], df_master['Keyword_Tieu_Cuc'] = zip(*df_master['Keyword'].apply(split_keywords))

# SẮP XẾP LẠI THỨ TỰ CỘT
column_final = [
    'Source_Platform', 'Candidate', 'Target_Position', 'Sentiment', 
    'Keyword_Tich_Cuc', 'Keyword_Tieu_Cuc', 'Field', 
    'Like_count', 'Content_Raw', 'URL'
]

df_master_final = df_master[column_final]

# XUẤT FILE
output_file = "Dataset_Candidate.csv"
df_master_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"Đã gộp thành công! Tổng cộng có {len(df_master_final)} dòng dữ liệu.")
print(f"File '{output_file}' đã sẵn sàng để làm báo cáo.")